# Semana 7 Día 1

Afinar un modelo de código abierto para predecir los precios de los productos

Consulte este cuaderno en Google Colab:

https://colab.research.google.com/drive/1dbATHAeHXd4CCHScgzsU-YYbcrHqiyqf?usp=sharing

## Predecir precios de productos

### Semana 7 Día 1

Introducción a LoRA y QLoRA

In [ ]:
# pip installs

!pip install -q datasets==2.21.0 requests torch peft bitsandbytes transformers==4.43.1 trl accelerate sentencepiece

In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from datetime import datetime

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
FINETUNED_MODEL = f"ed-donner/pricer-2024-09-13_13.04.39"

# Hyperparameters para Fine-Tuning QLoRA

LORA_R = 32
LORA_ALPHA = 64
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

### Inicia sesión en HuggingFace

Si aún no tienes una cuenta de HuggingFace, visita https://huggingface.co para registrarte y crear un token.

Luego, selecciona los secretos para este cuaderno haciendo clic en el ícono de la llave a la izquierda y agrega un nuevo secreto llamado `HF_TOKEN` con el valor como tu token.

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Probando diferentes Cuantizaciones


In [ ]:
# Cargar el modelo base sin cuantizar

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")

In [ ]:
print(f"Impacto en la memoria: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
base_model

## ¡Reinicia tu sesión!

Para cargar el siguiente modelo y limpiar la memoria caché del último modelo, ahora tendrás que ir a Runtime >> Reiniciar sesión y ejecutar las celdas iniciales (importaciones e inicio de sesión de HuggingFace) nuevamente.

Esto es para limpiar la GPU.

In [ ]:
# Cargar el modelo base en 8 bits

quant_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

In [ ]:
print(f"Impacto en Memoria: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
base_model

## ¡Reinicia tu sesión!

Para cargar el siguiente modelo y limpiar la memoria caché del último modelo, ahora tendrás que ir a Runtime >> Reiniciar sesión y ejecutar las celdas iniciales (importaciones e inicio de sesión de HuggingFace) nuevamente.

Esto es para limpiar la GPU.

In [ ]:
# Cargar el Tokenizer y el modelo Base en 4 bit

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

In [ ]:
print(f"Impacto en memoria: {base_model.get_memory_footprint() / 1e9:,.2f} GB")

In [ ]:
base_model

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)

In [ ]:
print(f"Impacto en memoria: {fine_tuned_model.get_memory_footprint() / 1e9:,.2f} GB")

In [ ]:
fine_tuned_model

In [ ]:
# Cada uno de los módulos de destino tiene 2 matrices de adaptador LoRA, llamadas lora_A y lora_B
# Están diseñadas para que los pesos se puedan adaptar sumando alpha * lora_A * lora_B
# Contemos la cantidad de pesos usando sus dimensiones:

# Vea las dimensiones de la matriz anterior
lora_q_proj = 4096 * 32 + 4096 * 32
lora_k_proj = 4096 * 32 + 1024 * 32
lora_v_proj = 4096 * 32 + 1024 * 32
lora_o_proj = 4096 * 32 + 4096 * 32

# Cada capa utiliza pues
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj

# Hay en total 32 capas
params = lora_layer * 32

# Por tanto el total en Mb es:
size = (params * 4) / 1_000_000

print(f"Número total de parámetros: {params:,} y tamaño {size:,.1f}MB")